# Download Mod File JSON 

In [ ]:
import os
import pandas as pd
import requests
import json
import re
from tqdm import tqdm

# Load dataset from Excel
df = pd.read_excel("../data/model_db_annotations.xlsx")

# Output JSON files
output_json = "../data/mod_files.json"
failed_json = "../data/failed_downloads.json"

# Lists to store mod file data and failed downloads
mod_files_data = []
failed_downloads = []

# Function to convert ModelDB URL to direct download URL
def get_direct_download_url(url):
    match = re.search(r"https://modeldb\.science/(\d+)\?tab=2&file=(.+)", url)
    if match:
        model_id, file_path = match.groups()
        return f"https://modeldb.science/getModelFile?model={model_id}&file={file_path}", file_path
    return None, None  # Return None if the URL doesn't match the expected pattern

# Function to fetch .mod file content and store in JSON
def fetch_mod_file(row_id, file_hash, raw_sha, count, url):
    direct_url, file_path = get_direct_download_url(url)
    
    # Default structure for JSON entry
    file_entry = {
        "row_id": row_id,
        "file_hash": file_hash,
        "raw_sha": raw_sha,
        "count": count,
        "url": url,
        "download_url": direct_url,
        "content": None,  # Default to None (indicating missing content)
        "error_code": None  # New field for failed downloads
    }

    if not direct_url:
        print(f"🚨 Skipping invalid URL: {url}")
        file_entry["error_code"] = "Invalid URL"
        failed_downloads.append(file_entry)
        mod_files_data.append(file_entry)  # Store even failed ones
        return

    try:
        response = requests.get(direct_url, timeout=10)
        response.raise_for_status()
        file_entry["content"] = response.text  # Store the downloaded content
    except requests.exceptions.HTTPError as http_err:
        file_entry["error_code"] = response.status_code  # Store specific HTTP error code
        print(f"❌ Failed to fetch {direct_url} - HTTP {response.status_code}: {http_err}")
        failed_downloads.append(file_entry)
    except requests.exceptions.RequestException as e:
        file_entry["error_code"] = "Request Error"
        print(f"❌ Failed to fetch {direct_url}: {e}")
        failed_downloads.append(file_entry)

    mod_files_data.append(file_entry)  # Store the file entry (success or failure)

# Select rows from 467 (corresponding to row_id 468) and take 11 rows
start = 0
n_rows = df.shape[0] 
df_subset = df.iloc[start:start+n_rows]

# Process selected files
for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Fetching .mod files"):
    fetch_mod_file(
        row["row_id"], 
        row["file_hash"], 
        row["raw_sha"], 
        row["count"],  
        row["url"]
    )

# Save all .mod data to JSON (including failed downloads)
with open(output_json, "w", encoding="utf-8") as json_file:
    json.dump(mod_files_data, json_file, indent=4)

# Save failed downloads separately for easy retrying
if failed_downloads:
    with open(failed_json, "w", encoding="utf-8") as json_file:
        json.dump(failed_downloads, json_file, indent=4)
    print(f"⚠️ Some downloads failed. Check {failed_json} for details.")

print(f"\n✅ All .mod files (including failures) saved in {output_json}")

# Download Mod-Files (Only Annotated Ion Channels)

In [1]:
import os
import pandas as pd
import requests
from urllib.parse import urlparse, parse_qs
from tqdm import tqdm  # Progress bar

In [2]:
json_df = pd.read_json("../data/raw/mod_files.json")

In [3]:
ant_channel = pd.read_csv("../data/pipeline/preprocessed.csv", usecols=["file_hash","new_subtype_label"]).query("new_subtype_label not in ['Z Neither','Receptor']")

In [4]:
def View(df, rows=None, cols=None, width=None):
    """Displays the first `rows` of the DataFrame like R's View() by adjusting Pandas settings."""
    
    # Show only the first `rows` of the DataFrame
    with pd.option_context(
        "display.max_rows", rows,  # Limit number of rows shown
        "display.max_columns", cols,  # Show all columns
        "display.max_colwidth", width,  # Show full column width
        "display.expand_frame_repr", False  # Prevent column wrapping
    ):
        display(df.head(rows))  # Show only the first `rows`

In [5]:
json_df2 = json_df.merge(ant_channel, how="inner", on="file_hash")

In [6]:
#before was (554, 9)
json_df2.shape

(721, 9)

In [7]:
json_df2["new_subtype_label"].value_counts()

new_subtype_label
I K (A-type)               93
I Na (Transient)           85
I K (Delayed Rectifier)    80
I Ca (HVA)                 72
I K (Ca-activated)         68
I H                        47
I K (M-type)               40
R Glutamate (NMDA)         39
I K (Rare)                 39
R GABA                     33
R Glutamate (AMPA)         31
I Ca (T-type LT)           31
I Na (Persistent)          22
I Other (Leak)             15
I Na (General)             15
I Other (Rare)             11
Name: count, dtype: int64

In [8]:
json_df2

,row_id,file_hash,raw_sha,count,url,download_url,content,error_code,new_subtype_label
0,1,f8be35d0c20d1b1f3de4c44323e1780ee24f06893b6364...,67b75245b9690ce219f34a5905425672dcb5c0661a885f...,6,https://modeldb.science/183300?tab=2&file=Shor...,https://modeldb.science/getModelFile?model=183...,TITLE Sodium persistent current\n\nCOMMENT\n ...,None,I Na (Persistent)
1,2,e97ca8a7f9734805832e5ae75442d19d3d7796b9a24190...,b45c13164f1eeaf4ba61ee8522304f7382fa0541831b1a...,1,https://modeldb.science/223649?tab=2&file=Altu...,https://modeldb.science/getModelFile?model=223...,TITLE K-A channel from Klee Ficker and Heinema...,None,I K (A-type)
2,4,606423f8f4a4f406f3387c7ee6f142bd121237272c347d...,81645a02d4dbf324cf8b82e8f31bda2ce46c3753f7a8ac...,1,https://modeldb.science/262115?tab=2&file=demo...,https://modeldb.science/getModelFile?model=262...,TITLE multiple GABAa receptors\n\nCOMMENT\n---...,None,R GABA
3,5,99713c0032634e96cc7cde2dce02d8aa2baf75ad4cc835...,5fe32f050754f534d60c7c126ecd805216089cd0594062...,1,https://modeldb.science/150288?tab=2&file=KimE...,https://modeldb.science/getModelFile?model=150...,:Interneuron Cells to Pyramidal Cells GABA wit...,None,R GABA
4,11,720e1c556b6f8f457c8910a66b7594378e3aad2e835b9c...,deba2eac5d4a238c4076a5155fbc5a568f321176ffac6e...,1,https://modeldb.science/144541?tab=2&file=Ih_c...,https://modeldb.science/getModelFile?model=144...,TITLE nax\r\n: Na current for axon. No slow in...,None,I Na (Transient)
...,...,...,...,...,...,...,...,...,...
716,1293,edf2b946f4b9b17e0050552f62c745b61bda2f0b90cbd7...,b3e7a5bf103a0ee404e862a93ad0b9f9309d5327f53ed0...,1,https://modeldb.science/168414?tab=2&file=Zhan...,https://modeldb.science/getModelFile?model=168...,TITLE HH sodium channel\r\n: Hodgkin - Huxley ...,None,I Na (Transient)
717,1294,9ce2c80b5bd5b8910d2524ec7e6a147d65384d633eb938...,869b219257f0053c7ae09c6270855077f172e9dd7b0386...,6,https://modeldb.science/267511?tab=2&file=S1_T...,https://modeldb.science/getModelFile?model=267...,TITLE CaGk\n: Calcium activated K channel.\n: ...,None,I K (Ca-activated)
718,1295,241fe560ad656781054be429af7c5cee507bae0a0383c8...,c931c738fbb10ebe53c43e3cd43fb5d169cee85b9d46f9...,1,https://modeldb.science/113446?tab=2&file=NEUR...,https://modeldb.science/getModelFile?model=113...,TITLE kir\nCOMMENT\npotassium inward rectifier...,None,I K (Rare)
719,1296,a453dc7a99d316fea476fc93808e4b8a939d176ec78ab9...,bbf252efb2311c0c0d89054987be1cfdc36ae9c84d36d8...,1,https://modeldb.science/85112?tab=2&file=baker...,https://modeldb.science/getModelFile?model=851...,: nav1p9.mod is the NaV1.9 Na+ current from\n:...,None,I Na (Persistent)


In [9]:
import os
import pandas as pd
import requests
from tqdm import tqdm

# Base output directory
base_dir = "../data/raw/nmodl"
os.makedirs(base_dir, exist_ok=True)

# Loop through the DataFrame rows
for _, row in tqdm(json_df2.iterrows(), total=len(json_df2), desc="Downloading MOD files"):
    try:
        url = row["download_url"]
        file_hash = row["file_hash"]

        # Download and save as filehash.mod
        response = requests.get(url)
        response.raise_for_status()

        output_path = os.path.join(base_dir, f"{file_hash}.mod")
        with open(output_path, "wb") as f:
            f.write(response.content)

    except Exception as e:
        print(f"Error downloading {url}: {e}")

# Download All Mod-Files

In [2]:
import os
import pandas as pd
import requests
from tqdm import tqdm

# Load JSON data
json_df = pd.read_json("../data/raw/mod_files.json")

# Base output directory
base_dir = "../data/raw/nmodl_all"
os.makedirs(base_dir, exist_ok=True)

# Log directory and file
log_dir = "../download_logs"
log_file = os.path.join(log_dir, "failed_downloads.log")
os.makedirs(log_dir, exist_ok=True)

# Loop through the DataFrame rows
for _, row in tqdm(json_df.iterrows(), total=len(json_df), desc="Downloading ALL MOD files"):
    try:
        url = row["download_url"]
        file_hash = row["file_hash"]

        # Download and save as filehash.mod
        response = requests.get(url)
        response.raise_for_status()

        output_path = os.path.join(base_dir, f"{file_hash}.mod")
        with open(output_path, "wb") as f:
            f.write(response.content)

    except Exception as e:
        print(f"Error downloading {url}: {e}")
        with open(log_file, "a") as log:
            log.write(f"{file_hash},{url},{e}\n")


Error downloading https://modeldb.science/getModelFile?model=58173&file=b05oct26/napRT03.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=58173&file=b05oct26/napRT03.mod


Error downloading https://modeldb.science/getModelFile?model=65218&file=nrntraub/mod/napf.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=65218&file=nrntraub/mod/napf.mod
Error downloading https://modeldb.science/getModelFile?model=38241&file=CA1_multi/mechanism/not-currently-used/it1.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=38241&file=CA1_multi/mechanism/not-currently-used/it1.mod


Error downloading None: Invalid URL 'None': No scheme supplied. Perhaps you meant https://None?


Error downloading https://modeldb.science/getModelFile?model=58173&file=b05oct26/intf.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=58173&file=b05oct26/intf.mod


Error downloading https://modeldb.science/getModelFile?model=97882&file=TOM/klb2002/kx.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=97882&file=TOM/klb2002/kx.mod


...

Error downloading https://modeldb.science/getModelFile?model=66276&file=nrntraub/mod/rand.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=66276&file=nrntraub/mod/rand.mod


Error downloading https://modeldb.science/getModelFile?model=116081&file=mod-Files/ca.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=116081&file=mod-Files/ca.mod


Error downloading https://modeldb.science/getModelFile?model=38229&file=kcum.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=38229&file=kcum.mod
Error downloading https://modeldb.science/getModelFile?model=38235&file=CoHCNS2000/coh.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=38235&file=CoHCNS2000/coh.mod


Error downloading https://modeldb.science/getModelFile?model=58173&file=b05oct26/cah.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=58173&file=b05oct26/cah.mod


Error downloading https://modeldb.science/getModelFile?model=65218&file=nrntraub/mod/kdr_fs.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=65218&file=nrntraub/mod/kdr_fs.mod


In [34]:
from pathlib import Path
import pandas as pd

# Define the folder path
mod5k_dir = Path("../data/raw/mod-5k")

# Get list of all files (adjust pattern if needed)
file_list = [f.name for f in mod5k_dir.glob("*") if f.is_file()]

# Create DataFrame
df_files = pd.DataFrame({"file_name": file_list})
df_files["file_hash"] = df_files["file_name"].str.replace(".mod", "", regex=False)


In [44]:
df_files.shape

(4959, 2)

In [35]:
link_df = df_files.merge(json_df, how="inner", on="file_hash")

In [40]:
link_df2 = link_df[["file_hash","url"]]

In [43]:
link_df2.to_csv("../data/mod_files_key.csv", index=False)